# LangGraph with AgentCore Memory Manager (Long-term Memory)

## Introduction

This notebook demonstrates how to build an intelligent nutrition assistant using LangGraph framework integrated with Amazon Bedrock AgentCore Memory capabilities. We'll use the **MemoryManager**, **MemorySessionManager**, and **MemorySession** APIs to implement **long-term memory** retention across multiple conversation sessions - allowing an agent to extract and recall user preferences, dietary restrictions, and contextual information from past interactions.

## Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Long-term Conversational                                                        |
| Agent usecase       | Nutrition Assistant                                                              |
| Agentic Framework   | LangGraph                                                                        |
| LLM model           | Anthropic Claude Sonnet 3.7                                                     |
| Tutorial components | AgentCore MemoryManager, MemorySessionManager, Custom Memory Strategies, Pre/Post Model Hooks |
| Example complexity  | Intermediate                                                                     |

You'll learn to:
- Set up AgentCore Memory using MemoryManager with typed strategy objects
- Use MemorySessionManager for session-based memory operations
- Implement pre/post model hooks with memory retrieval
- Build a nutrition assistant that remembers user preferences across sessions
- Configure custom memory extraction and consolidation with execution roles

### Scenario Context

In this example, we'll create a **Nutrition Assistant** that can remember user context across multiple conversations, including dietary restrictions, favorite foods, cooking preferences, and health goals. The agent will automatically extract and store user preferences from conversations using AgentCore's MemoryManager, then retrieve relevant context for future interactions through MemorySession to provide personalized nutrition advice.

## Architecture

<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites

- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models
- AgentCore Memory Manager SDK

Let's get started by setting up our environment!

In [ ]:
# Install necessary libraries
%pip install -qr requirements.txt

In [ ]:
import os
import logging
import json
import boto3
import sys
from datetime import datetime
from typing import Dict, List
from botocore.exceptions import ClientError

# Import LangGraph and LangChain components
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
import uuid

# Import AgentCore Memory components
from langgraph_checkpoint_aws import AgentCoreMemoryStore
from langgraph.checkpoint.memory import InMemorySaver

# Import MemoryManager and session management
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    CustomUserPreferenceStrategy, ExtractionConfig, ConsolidationConfig
)
from bedrock_agentcore.memory.session import MemorySessionManager, MemorySession
from bedrock_agentcore.memory.constants import MessageRole, RetrievalConfig
from bedrock_agentcore.memory.models import MemoryRecord

# Import custom memory prompts
current_dir = os.path.dirname(os.path.abspath(os.getcwd()))
sys.path.append(current_dir) 
from custom_memory_prompts import consolidation_prompt, extraction_prompt

# Configuration
region = os.getenv('AWS_REGION', 'us-east-1')
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
ACTOR_ID = "nutrition-user-1"
SESSION_ID = f"nutrition_{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("nutrition-agent")

## Step 1: Create IAM Execution Role for Custom Memory Strategies

Custom memory strategies in AgentCore Memory require an execution role that allows the service to invoke Bedrock models for extraction and consolidation. This role is essential when using `CustomUserPreferenceStrategy` with the MemoryManager.

The execution role needs:
- Trust policy allowing `bedrock-agentcore.amazonaws.com` to assume the role
- Permissions to invoke Bedrock foundation models
- Proper resource and condition constraints for security

In [ ]:
def create_memory_execution_role():
    """Create IAM role for AgentCore Memory custom strategies with required permissions"""
    iam_client = boto3.client('iam', region_name=region)
    
    # Get current AWS account ID
    sts_client = boto3.client('sts', region_name=region)
    account_id = sts_client.get_caller_identity()['Account']
    
    role_name = "NutritionAgentMemoryExecutionRole"
    role_arn = f"arn:aws:iam::{account_id}:role/{role_name}"
    
    # Trust policy for AgentCore Memory service
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": ["bedrock-agentcore.amazonaws.com"]
                },
                "Action": "sts:AssumeRole",
                "Condition": {
                    "StringEquals": {
                        "aws:SourceAccount": account_id
                    },
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{region}:{account_id}:*"
                    }
                }
            }
        ]
    }
    
    # Permissions policy for Bedrock model invocation
    permissions_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                    "bedrock:InvokeModelWithResponseStream"
                ],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    "arn:aws:bedrock:*:*:inference-profile/*"
                ],
                "Condition": {
                    "StringEquals": {
                        "aws:ResourceAccount": account_id
                    }
                }
            }
        ]
    }
    
    try:
        # Check if role already exists
        try:
            existing_role = iam_client.get_role(RoleName=role_name)
            logger.info(f"✅ IAM execution role already exists: {role_arn}")
            return role_arn
        except ClientError as e:
            if e.response['Error']['Code'] != 'NoSuchEntity':
                raise
        
        # Create the role
        logger.info(f"Creating IAM execution role: {role_name}")
        iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="Execution role for AgentCore Memory custom strategies",
            Tags=[
                {'Key': 'Purpose', 'Value': 'NutritionAgentMemory'},
                {'Key': 'Application', 'Value': 'LangGraphNutritionAssistant'}
            ]
        )
        
        # Attach the permissions policy
        policy_name = "NutritionAgentBedrockAccess"
        iam_client.put_role_policy(
            RoleName=role_name,
            PolicyName=policy_name,
            PolicyDocument=json.dumps(permissions_policy)
        )
        
        logger.info(f"✅ Successfully created IAM execution role: {role_arn}")
        logger.info(f"   - Trust policy: AgentCore Memory service can assume this role")
        logger.info(f"   - Permissions: bedrock:InvokeModel for custom strategy processing")
        
        return role_arn
        
    except ClientError as e:
        error_code = e.response['Error']['Code']
        if error_code == 'AccessDenied':
            logger.error("❌ Access denied creating IAM role. Required permissions:")
            logger.error("   - iam:CreateRole, iam:PutRolePolicy, iam:GetRole")
        else:
            logger.error(f"❌ Failed to create IAM role: {e}")
        raise
    except Exception as e:
        logger.error(f"❌ Unexpected error creating IAM role: {e}")
        raise

# Create the execution role for custom memory strategies
try:
    MEMORY_EXECUTION_ROLE_ARN = create_memory_execution_role()
    logger.info(f"✅ Memory execution role ready: {MEMORY_EXECUTION_ROLE_ARN}")
except Exception as e:
    logger.error(f"❌ Failed to create memory execution role: {e}")
    raise

## Step 2: Initialize MemoryManager and Create Memory Resource

We'll use the **MemoryManager** to create and manage our nutrition assistant's memory resource. The MemoryManager provides a high-level interface for memory operations and supports typed strategy objects for better type safety and IDE support.

### Memory Strategy Configuration

For our nutrition assistant, we'll use a **CustomUserPreferenceStrategy** that:
- Extracts nutrition preferences from conversations using custom prompts
- Consolidates similar preferences to avoid duplication
- Stores preferences in organized namespaces by user ID

In [ ]:
# Initialize MemoryManager
memory_manager = MemoryManager(region_name=region)
memory_name = "NutritionAssistantMemory"

logger.info(f"✅ MemoryManager initialized for region: {region}")

# Define memory strategy using typed CustomUserPreferenceStrategy
nutrition_preference_strategy = CustomUserPreferenceStrategy(
    name="NutritionPreferences",
    description="Captures user food preferences, dietary restrictions, and nutrition goals",
    extraction_config=ExtractionConfig(
        append_to_prompt=extraction_prompt,
        model_id=MODEL_ID
    ),
    consolidation_config=ConsolidationConfig(
        append_to_prompt=consolidation_prompt,
        model_id=MODEL_ID
    ),
    namespaces=["/{actorId}/preferences"]
)

logger.info(f"✅ Configured CustomUserPreferenceStrategy:")
logger.info(f"   - Name: {nutrition_preference_strategy.name}")
logger.info(f"   - Description: {nutrition_preference_strategy.description}")

# Create memory resource using MemoryManager
memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[nutrition_preference_strategy],
    description="Long-term memory for nutrition assistant with user preference extraction",
    event_expiry_days=90,  # Memories expire after 90 days
    memory_execution_role_arn=MEMORY_EXECUTION_ROLE_ARN  # Required for custom strategies
)

memory_id = memory.id
logger.info(f"✅ Successfully created/retrieved memory with MemoryManager:")
logger.info(f"   Memory ID: {memory_id}")
logger.info(f"   Memory Name: {memory.name}")

## Step 3: Initialize MemorySessionManager and Create MemorySession

The **MemorySessionManager** provides session-based memory operations, making it easier to manage conversations for specific users. We'll create a **MemorySession** for our nutrition assistant user that handles all memory operations within the context of that user's sessions.

### Benefits of Session-Based Memory Management

- **Simplified API**: No need to pass actor_id and session_id repeatedly
- **Context Management**: Automatic handling of user context and session boundaries
- **Type Safety**: Better integration with typed memory operations
- **Error Handling**: Error handling for session-specific operations

In [ ]:
# Initialize MemorySessionManager for the created memory
session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)

# Create a MemorySession for the nutrition assistant user
nutrition_session = session_manager.create_memory_session(
    actor_id=ACTOR_ID,
    session_id=SESSION_ID
)

logger.info(f"✅ MemorySessionManager initialized for memory: {memory_id}")

## Step 4: Initialize AgentCore Memory Store and LLM

Now we'll initialize the **AgentCoreMemoryStore** that integrates with LangGraph, and set up our language model. The memory store will work with our MemoryManager-created memory resource to provide integration between LangGraph and AgentCore Memory.

In [ ]:
# Initialize the AgentCore Memory Store for LangGraph integration
store = AgentCoreMemoryStore(memory_id=memory_id, region_name=region)

# Initialize Bedrock LLM
llm = init_chat_model(MODEL_ID, model_provider="bedrock_converse", region_name=region)

logger.info(f"✅ AgentCoreMemoryStore initialized with memory ID: {memory_id}")
logger.info(f"✅ Bedrock LLM initialized: {MODEL_ID}")

## Step 5: Implement Memory Hooks

We'll create pre and post model hooks that leverage our MemorySession for memory operations:

### Pre-Model Hook Features
- **Semantic Search**: Retrieves relevant user preferences based on current query
- **Relevance Scoring**: Filters memories by relevance score to ensure quality
- **Context Injection**: Adds retrieved preferences as context before LLM invocation
- **Namespace Resolution**: Properly resolves user-specific namespaces

### Post-Model Hook Features
- **Automatic Storage**: Saves conversations for long-term memory extraction
- **Session Management**: Uses MemorySession for cleaner API calls
- **Error Handling**: Error handling for memory operations

### Memory Processing Workflow

1. **Message Storage**: Conversations are saved to AgentCore Memory with actor_id and session_id
2. **Background Processing**: Custom strategy processes conversations to extract nutrition preferences
3. **Preference Storage**: Extracted preferences are stored in the `{actorId}/preferences` namespace
4. **Context Retrieval**: Future conversations search and retrieve relevant preferences for personalization

In [ ]:
def pre_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Pre-model hook with memory retrieval using MemorySession"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    
    # Save the latest human message to memory store
    namespace = (actor_id, thread_id)
    messages = state.get("messages", [])
    
    # Find and save the last human message
    last_human_message = None
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage):
            last_human_message = msg
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            break
    
    if not last_human_message:
        return {"messages": messages}
    
    # Preference retrieval with relevance scoring
    try:
        # Define retrieval configuration
        retrieval_config = RetrievalConfig(
            top_k=5,  # Get top 5 most relevant preferences
            relevance_score=0.3  # Minimum relevance threshold
        )
        
        # Search for relevant user preferences using semantic search
        user_preferences_namespace = (actor_id, "preferences")
        preferences = store.search(
            user_preferences_namespace, 
            query=last_human_message.content, 
            limit=retrieval_config.top_k
        )
        
        # Filter preferences by relevance score if available
        relevant_preferences = []
        for pref in preferences:
            # Check if preference has a relevance score and meets threshold
            if hasattr(pref, 'score') and pref.score is not None:
                if pref.score >= retrieval_config.relevance_score:
                    relevant_preferences.append(pref)
            else:
                # Include preferences without scores (backward compatibility)
                relevant_preferences.append(pref)
        
        # Create context message if relevant preferences found
        if relevant_preferences:
            context_items = []
            for i, pref in enumerate(relevant_preferences[:3], 1):  # Limit to top 3
                pref_text = str(pref.value) if hasattr(pref, 'value') else str(pref)
                score_text = f" (Score: {pref.score:.2f})" if hasattr(pref, 'score') and pref.score is not None else ""
                context_items.append(f"{i}. {pref_text}{score_text}")
            
            context_message = AIMessage(
                content=f"[User Nutrition Context - Retrieved {len(relevant_preferences)} relevant preferences:\n" + 
                       "\n".join(context_items) + "]"
            )
            
            logger.info(f"✅ Pre-model hook: Retrieved {len(relevant_preferences)} relevant preferences")
            
            # Insert context before the last human message
            return {"messages": messages[:-1] + [context_message, messages[-1]]}
        else:
            logger.info("ℹ️ No relevant preferences found for current query")
            
    except Exception as e:
        logger.error(f"❌ Error in pre-model hook: {e}")
        # Continue without context if retrieval fails
    
    return {"messages": messages}

def post_model_hook(state, config: RunnableConfig, *, store: BaseStore):
    """Post-model hook with session-based memory storage"""
    actor_id = config["configurable"]["actor_id"]
    thread_id = config["configurable"]["thread_id"]
    
    # Save the latest AI message to memory store
    namespace = (actor_id, thread_id)
    messages = state.get("messages", [])
    
    # Find and save the last AI message
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and not msg.content.startswith("[User"):
            # Skip context messages, save only actual AI responses
            store.put(namespace, str(uuid.uuid4()), {"message": msg})
            logger.info("✅ Post-model hook: Saved AI response to memory")
            break
    
    return {"messages": messages}

logger.info("✅ Memory hooks defined with MemorySession")

## Step 6: Create the LangGraph Agent with Memory Integration

Now we'll create our nutrition assistant agent using LangGraph's `create_react_agent` with our memory hooks. The agent will automatically:

- **Retrieve Context**: Search for relevant nutrition preferences before each response
- **Store Conversations**: Save interactions for long-term memory extraction
- **Provide Personalized Advice**: Use retrieved preferences to tailor nutrition recommendations
- **Learn Over Time**: Continuously extract and consolidate user preferences

In [ ]:
# Create the nutrition assistant agent with memory capabilities
graph = create_react_agent(
    llm,
    store=store,  # AgentCore Memory Store
    tools=[],  
    checkpointer=InMemorySaver(),  # For conversation state management
    pre_model_hook=pre_model_hook,  # preference retrieval
    post_model_hook=post_model_hook  # conversation storage
)

logger.info("✅ LangGraph nutrition assistant created with memory integration")
logger.info("   - Memory Store: AgentCore Memory with MemoryManager")
logger.info("   - Pre-hook: Preference retrieval with relevance scoring")
logger.info("   - Post-hook: Session-based conversation storage")
logger.info("   - Checkpointer: InMemorySaver for conversation state")

## Step 7: Configure Agent Runtime with Session Management

We'll configure the agent with unique identifiers that correspond to our MemorySession setup. These IDs are essential for proper memory organization and retrieval.

### Configuration Details

- **thread_id**: Maps to AgentCore Memory session_id for conversation tracking
- **actor_id**: Maps to AgentCore Memory actor_id for user identification

These IDs align with our MemorySession configuration

In [ ]:
# Configure runtime with session identifiers
config = {
    "configurable": {
        "thread_id": SESSION_ID,  # Maps to AgentCore Memory session_id
        "actor_id": ACTOR_ID,     # Maps to AgentCore Memory actor_id
    }
}

logger.info(f"✅ Agent runtime configured:")
logger.info(f"   Thread ID (Session): {SESSION_ID}")
logger.info(f"   Actor ID (User): {ACTOR_ID}")
logger.info(f"   Memory Integration: MemorySession-based operations")

## Step 8: Test the Nutrition Assistant

Let's test our nutrition assistant by having a conversation about food preferences. The agent will automatically extract and store user preferences using our CustomUserPreferenceStrategy for future personalization.

### What Happens During This Test

1. **User Input**: We provide nutrition-related information and questions
2. **Memory Storage**: Pre/post hooks save the conversation to AgentCore Memory
3. **Background Processing**: Custom strategy extracts nutrition preferences
4. **Preference Storage**: Extracted preferences are stored in user-specific namespace
5. **Future Retrieval**: Subsequent conversations can access these preferences

In [ ]:
# Helper function to run agent
def run_nutrition_agent(query: str, config: RunnableConfig):
    """Run the nutrition assistant with logging and output formatting"""
    logger.info(f"🍎 Running nutrition assistant with query: {query[:50]}...")
    
    printed_ids = set()
    events = graph.stream(
        {"messages": [{"role": "user", "content": query}]},
        config,
        stream_mode="values",
    )
    
    for event in events:
        if "messages" in event:
            for msg in event["messages"]:
                # Check if we've already printed this message
                if id(msg) not in printed_ids:
                    msg.pretty_print()
                    printed_ids.add(id(msg))
    
    logger.info("✅ Nutrition assistant response completed")

# Test 1: Initial conversation with nutrition preferences and goals
initial_prompt = """
Hey there! I'm cooking one of my favorite meals tonight - salmon with rice and vegetables. 
I really focus on healthy eating because I have a weightlifting competition coming up and need 
to maintain good macros. I love Mediterranean flavors and try to get plenty of protein and vitamins. 
What can I add to this dish to make it taste better and also improve the protein and vitamins I get?
"""

run_nutrition_agent(initial_prompt, config)

## Step 9: Verify Memory Storage and Extraction

Let's verify that our CustomUserPreferenceStrategy is working correctly by checking what preferences have been extracted and stored. This demonstrates the power of AgentCore Memory's background processing.

**Note**: AgentCore Memory processes conversations in the background, so it may take a few moments for preferences to be extracted and available for retrieval.

In [ ]:
# Check what preferences have been extracted and stored
def check_stored_preferences():
    """Check what nutrition preferences have been extracted and stored"""
    try:
        # Search the user preferences namespace
        search_namespace = (ACTOR_ID, "preferences")
        preferences = store.search(search_namespace, query="nutrition food preferences", limit=10)
        
        logger.info(f"🔍 Checking stored preferences in namespace: {search_namespace}")
        
        if preferences:
            logger.info(f"✅ Found {len(preferences)} stored preferences:")
            for i, pref in enumerate(preferences, 1):
                pref_text = str(pref.value) if hasattr(pref, 'value') else str(pref)
                score_text = f" (Score: {pref.score:.2f})" if hasattr(pref, 'score') and pref.score is not None else ""
                logger.info(f"   {i}. {pref_text[:100]}...{score_text}")
                print(f"   {i}. {pref_text[:150]}...{score_text}")
        else:
            logger.info("ℹ️ No preferences found yet. Memory extraction may still be processing.")
            print("No preferences extracted yet. This is normal - AgentCore Memory processes conversations in the background.")
            print("Try running this cell again in a few moments.")
            
    except Exception as e:
        logger.error(f"❌ Error checking stored preferences: {e}")
        print(f"Error checking preferences: {e}")

check_stored_preferences()

## Step 10: Test Memory Retrieval with New Session

Now let's test the memory retrieval capabilities by starting a new session and asking for nutrition advice. The agent should be able to access the previously stored preferences and provide personalized recommendations.

### Memory Retrieval Process

1. **New Session**: We create a new session ID to simulate a different conversation
2. **Semantic Search**: The pre-model hook searches for relevant preferences
3. **Context Injection**: Retrieved preferences are added as context
4. **Personalized Response**: The agent uses this context to provide tailored advice

In [ ]:
# Create a new session to test memory retrieval across sessions
new_session_id = f"nutrition_session_2_{datetime.now().strftime('%H%M%S')}"
new_config = {
    "configurable": {
        "thread_id": new_session_id,  # New session ID
        "actor_id": ACTOR_ID,         # Same actor ID to access stored preferences
    }
}

logger.info(f"🔄 Testing memory retrieval with new session: {new_session_id}")

# Test query that should trigger preference retrieval
memory_test_prompt = """
It's a new day! What should I make for dinner tonight? I want something that aligns 
with my fitness goals and taste preferences.
"""

run_nutrition_agent(memory_test_prompt, new_config)

## Step 11: Advanced Memory Operations with MemorySession

Memory operations using MemorySession directly. This shows how you can programmatically interact with the long term memory beyond the automatic hooks.

### Direct Memory Operations

- **Search Long-term Memories**: Direct search of extracted preferences
- **Add Custom Memories**: Programmatically add specific preferences
- **Session Management**: Leverage session-based operations for cleaner code

In [ ]:
# Demonstrate direct memory operations using MemorySession
def demonstrate_memory_session_operations():
    """Demonstrate advanced memory operations using MemorySession"""
    try:
        logger.info("🧠 Demonstrating MemorySession operations:")
        
        # 1. Search long-term memories directly
        logger.info("1. Searching long-term memories for 'protein'...")
        protein_memories = nutrition_session.search_long_term_memories(
            query="protein requirements fitness",
            namespace_prefix=f"/{ACTOR_ID}/preferences",
            top_k=3
        )
        
        if protein_memories:
            logger.info(f"   Found {len(protein_memories)} protein-related memories")
            for i, memory in enumerate(protein_memories, 1):
                content = memory.get('content', {}).get('text', 'No content')
                score = memory.get('score', 'N/A')
                print(f"   {i}. (Score: {score}) {content[:100]}...")
        else:
            logger.info("   No protein-related memories found")
        
        # 2. Get session information
        logger.info("2. Session information:")
        logger.info(f"   Session Type: {type(nutrition_session).__name__}")
        
        # 3. Search for Mediterranean preferences
        logger.info("3. Searching for Mediterranean cuisine preferences...")
        mediterranean_memories = nutrition_session.search_long_term_memories(
            query="Mediterranean flavors cuisine",
            namespace_prefix=f"/{ACTOR_ID}/preferences",
            top_k=2
        )
        
        if mediterranean_memories:
            logger.info(f"   Found {len(mediterranean_memories)} Mediterranean-related memories")
            for i, memory in enumerate(mediterranean_memories, 1):
                content = memory.get('content', {}).get('text', 'No content')
                score = memory.get('score', 'N/A')
                print(f"   {i}. (Score: {score}) {content[:100]}...")
        else:
            logger.info("   No Mediterranean-related memories found")
            
    except Exception as e:
        logger.error(f"❌ Error in memory session operations: {e}")
        print(f"Error: {e}")

demonstrate_memory_session_operations()

## Step 12: Test Continuous Learning

Let's add more nutrition information and see how the agent continues to learn and adapt its recommendations based on accumulated preferences.

### Continuous Learning Process

1. **New Information**: Provide additional dietary preferences and restrictions
2. **Extraction**: CustomUserPreferenceStrategy extracts new preferences
3. **Consolidation**: Similar preferences are consolidated to avoid duplication
4. **Enhanced Context**: Future responses use the expanded preference base

In [ ]:
# Add more nutrition information to test continuous learning
additional_info_prompt = """
I should mention that I'm also trying to reduce my sugar intake and I'm lactose intolerant, 
so I avoid dairy products. I really enjoy Asian cuisine too, especially Thai and Japanese dishes. 
I'm also interested in plant-based proteins since I'm trying to eat more sustainably. 
Can you suggest a meal plan for tomorrow that considers all my preferences?
"""

logger.info("🌱 Testing continuous learning with additional dietary information")
run_nutrition_agent(additional_info_prompt, new_config)

## Step 13: Final Memory Verification

Let's do a final check of our stored preferences to see how the CustomUserPreferenceStrategy has captured and organized all the nutrition information from our conversations.

In [ ]:
# Final check of stored preferences
def final_memory_verification():
    """Verification of all stored nutrition preferences"""
    logger.info("🔍 Final memory verification:")
    
    # Check different types of preferences
    search_queries = [
        ("dietary restrictions allergies", "Dietary Restrictions & Allergies"),
        ("cuisine preferences flavors", "Cuisine Preferences"),
        ("fitness goals protein", "Fitness & Nutrition Goals"),
        ("food preferences cooking", "General Food Preferences")
    ]
    
    total_preferences = 0
    
    for query, category in search_queries:
        try:
            search_namespace = (ACTOR_ID, "preferences")
            preferences = store.search(search_namespace, query=query, limit=5)
            
            if preferences:
                logger.info(f"\n📋 {category} ({len(preferences)} items):")
                print(f"\n📋 {category}:")
                
                for i, pref in enumerate(preferences, 1):
                    pref_text = str(pref.value) if hasattr(pref, 'value') else str(pref)
                    score_text = f" (Score: {pref.score:.2f})" if hasattr(pref, 'score') and pref.score is not None else ""
                    print(f"   {i}. {pref_text[:120]}...{score_text}")
                
                total_preferences += len(preferences)
            else:
                logger.info(f"\n📋 {category}: No specific preferences found")
                
        except Exception as e:
            logger.error(f"❌ Error searching {category}: {e}")
    
    logger.info(f"\n✅ Memory verification complete. Total preference items found: {total_preferences}")
    print(f"\n✅ Total nutrition preferences captured: {total_preferences}")
    
    if total_preferences > 0:
        print("\n🎉 Success! The nutrition assistant has successfully learned and stored your preferences.")
        print("These preferences will be used to provide personalized nutrition advice in future conversations.")
    else:
        print("\n⏳ Preferences may still be processing. AgentCore Memory extracts preferences in the background.")

final_memory_verification()

## Summary and Takeaways

Congratulations! You've successfully built a nutrition assistant using LangGraph with AgentCore Memory Manager integration. Here's what we accomplished:

### **Implementation**

- **MemoryManager**: High-level memory resource management with typed strategies
- **MemorySessionManager**: Session-based memory operations for cleaner API usage
- **MemorySession**: User-specific memory context management
- **CustomUserPreferenceStrategy**: Specialized nutrition preference extraction and consolidation
- **Memory Hooks**: Pre/post model hooks with relevance scoring

### **Memory Capabilities**

- **Automatic Extraction**: Conversations are automatically processed to extract nutrition preferences
- **Semantic Search**: Relevant preferences are retrieved based on current conversation context
- **Preference Consolidation**: Similar preferences are merged to avoid duplication
- **Cross-Session Memory**: Preferences persist and are accessible across different conversation sessions
- **Relevance Scoring**: Memory retrieval uses scoring to ensure quality context injection

### **Key features**

1. **Personalized Nutrition Advice**: Agent provides tailored recommendations based on learned preferences
2. **Continuous Learning**: System improves recommendations as it learns more about user preferences
3. **Session Management**: Clean separation between different conversation sessions while maintaining user context
4. **Type Safety**: Typed strategy objects provide better IDE support and error prevention

## Optional: Cleanup Resources

If you want to clean up the memory resources created during this tutorial, you can use the following code. **Note**: This will delete all stored preferences and conversation history.

In [ ]:
# Uncomment to delete the memory resource and IAM role
# WARNING: This will delete all stored preferences and conversation history

# def cleanup_resources():
#     """Clean up memory resources and IAM role"""
#     try:
#         # Delete memory resource
#         logger.info(f"Deleting memory resource: {memory_id}")
#         memory_manager.delete_memory(memory_id)
#         logger.info("✅ Memory resource deleted")
        
#         # Delete IAM role
#         iam_client = boto3.client('iam', region_name=region)
#         role_name = "NutritionAgentMemoryExecutionRole"
        
#         # Delete role policy first
#         try:
#             iam_client.delete_role_policy(
#                 RoleName=role_name,
#                 PolicyName="NutritionAgentBedrockAccess"
#             )
#         except ClientError:
#             pass  # Policy might not exist
        
#         # Delete role
#         iam_client.delete_role(RoleName=role_name)
#         logger.info("✅ IAM role deleted")
        
#     except Exception as e:
#         logger.error(f"❌ Error during cleanup: {e}")

# cleanup_resources()